# RHEAR E03 — Real-World Speech-Enhancement DatasetBuilds the L1 training dataset from **real recordings**: real clean speech, realenvironmental noise, real military noise, real measured room impulse responses.**This notebook does not train anything.** It produces the dataset and theDATASET REPORT that must be inspected before any training run.Provenance is tracked explicitly throughout:| tag | meaning ||---|---|| `real recording` | unmodified audio from a public corpus || `synthetically mixed from real recordings` | real speech + real noise combined by this pipeline || `synthetically generated` | produced by a signal generator — **not used here** |Runtime: ~25 min for the download (cached to Drive afterwards) + ~15 min to build.

## 0. Setup and Drive cacheEverything is cached under `MyDrive/RHEAR/E03` so a fresh session re-downloads nothing.

In [ ]:
import os, sys, subprocess, json, timeIN_COLAB = "google.colab" in sys.modulesif IN_COLAB:    from google.colab import drive    drive.mount("/content/drive")    ROOT = "/content/drive/MyDrive/RHEAR/E03"else:    ROOT = os.path.expanduser("~/RHEAR/E03")CORPORA = os.path.join(ROOT, "corpora")OUT     = os.path.join(ROOT, "dataset")os.makedirs(CORPORA, exist_ok=True); os.makedirs(OUT, exist_ok=True)print("ROOT:", ROOT)subprocess.run([sys.executable, "-m", "pip", "-q", "install", "soundfile"], check=False)

In [ ]:
# The pipeline code. In Colab, clone the repo; locally it is already on disk.CODE = os.path.join(ROOT, "code")if IN_COLAB and not os.path.isdir(os.path.join(CODE, "rhear_data")):    os.makedirs(CODE, exist_ok=True)    print("Upload PS#2/E03/rhear_data/ to", CODE, "or clone the project repo there.")sys.path.insert(0, CODE if IN_COLAB else os.path.dirname(os.getcwd()) + "/E03")sys.path.insert(0, "E03")from rhear_data import registry, build, report, manifest, splitsprint("pipeline loaded")

## 1. Datasets and licences — verified, not assumedEvery licence below was checked against the dataset's own page or licence file.**Policy.** RHEAR is a DRDO/iDEX project with a production path, so the defaultbuild uses only attribution-style licences (CC BY 4.0 / CC0 / US Public Domain /Apache 2.0). Two widely used corpora are **deliberately excluded**:* **ESC-50** and **UrbanSound8K** — CC BY-NC 3.0. A non-commercial term  contaminates downstream product use.* **DEMAND** — CC BY-SA 3.0. Share-alike is viral: a dataset derived from it must  itself be CC BY-SA.**MAD is used as a military-noise source only.** It is a classification corpuswith no clean-speech pairing, so it is never treated as an enhancement pair, andits `communication` class (which contains speech) is excluded from the noise poolso it cannot contaminate the target.

In [ ]:
rows = registry.licence_table()w = [max(len(str(r[i])) for r in rows) for i in range(6)]for j, r in enumerate(rows):    print("  " + "  ".join(str(r[i]).ljust(w[i]) for i in range(6)))    if j == 0: print("  " + "  ".join("-"*w[i] for i in range(6)))print()print("derived dataset licence:", registry.derived_licence(registry.SMALL_BUILD))

## 2. DownloadChoose a build profile. `small` is enough to validate the pipeline and to train afirst GTCRN-class baseline; `full` adds LibriSpeech train-clean-100 (251 speakers).| profile | download | speakers ||---|---|---|| `small` | ~12 GB | 80 (dev-clean + test-clean) || `full` | ~19 GB | 331 |

In [ ]:
PROFILE = "small"     # "small" or "full"URLS = { "dev-clean":       "https://www.openslr.org/resources/12/dev-clean.tar.gz", "test-clean":      "https://www.openslr.org/resources/12/test-clean.tar.gz", "musan":           "https://www.openslr.org/resources/17/musan.tar.gz", "rirs":            "https://www.openslr.org/resources/28/rirs_noises.zip",}if PROFILE == "full":    URLS["train-clean-100"] = "https://www.openslr.org/resources/12/train-clean-100.tar.gz"def fetch(name, url):    dest = os.path.join(CORPORA, os.path.basename(url))    if os.path.exists(dest) and os.path.getsize(dest) > 10_000_000:        print(f"  cached  {name}  ({os.path.getsize(dest)/1e9:.2f} GB)"); return dest    print(f"  downloading {name} ...")    subprocess.run(["wget", "-q", "--show-progress", "-O", dest, url], check=True)    return destpaths = {k: fetch(k, u) for k, u in URLS.items()}

In [ ]:
# MAD: hosted on Kaggle. Provide kaggle.json once; it is then cached in Drive.MAD_DIR = os.path.join(CORPORA, "mad")if not os.path.isdir(MAD_DIR):    kj = os.path.join(ROOT, "kaggle.json")    if os.path.exists(kj):        os.makedirs(os.path.expanduser("~/.kaggle"), exist_ok=True)        subprocess.run(["cp", kj, os.path.expanduser("~/.kaggle/kaggle.json")])        os.chmod(os.path.expanduser("~/.kaggle/kaggle.json"), 0o600)        subprocess.run([sys.executable, "-m", "pip", "-q", "install", "kaggle"])        subprocess.run(["kaggle", "datasets", "download", "-d",                        "junewookim/mad-dataset-military-audio-dataset",                        "-p", CORPORA, "--unzip"], check=False)    else:        print("MAD not fetched: put kaggle.json at", kj)        print("  (or download manually from github.com/kaen2891/military_audio_dataset)")        print("  The build runs without MAD, but then has NO military noise --")        print("  the report will say so and the dataset is not fit for the PS.")

In [ ]:
# Extract (idempotent)def extract(p, marker):    if os.path.isdir(os.path.join(CORPORA, marker)):        print("  already extracted:", marker); return    print("  extracting", os.path.basename(p))    if p.endswith(".zip"):        subprocess.run(["unzip", "-q", "-o", p, "-d", CORPORA], check=True)    else:        subprocess.run(["tar", "-xzf", p, "-C", CORPORA], check=True)extract(paths["dev-clean"],  "LibriSpeech/dev-clean")extract(paths["test-clean"], "LibriSpeech/test-clean")extract(paths["musan"],      "musan")extract(paths["rirs"],       "RIRS_NOISES")if PROFILE == "full":    extract(paths["train-clean-100"], "LibriSpeech/train-clean-100")print("done")

## 3. Verify the raw corporaSample rate, channels and duration are checked before anything is built.

In [ ]:
import soundfile as sf, numpy as np, glob, randomdef probe(pattern, n=200, label=""):    fs_set, ch_set, durs = set(), set(), []    files = glob.glob(pattern, recursive=True)    for p in random.Random(0).sample(files, min(n, len(files))):        try:            i = sf.info(p)            fs_set.add(i.samplerate); ch_set.add(i.channels); durs.append(i.duration)        except Exception: pass    print(f"  {label:<22} files={len(files):6d}  fs={sorted(fs_set)}  ch={sorted(ch_set)}"          f"  dur {np.min(durs):.1f}-{np.max(durs):.1f}s (median {np.median(durs):.1f})"          if durs else f"  {label}: none found")    return len(files)probe(os.path.join(CORPORA,"LibriSpeech/dev-clean/**/*.flac"), label="LibriSpeech dev-clean")probe(os.path.join(CORPORA,"LibriSpeech/test-clean/**/*.flac"), label="LibriSpeech test-clean")probe(os.path.join(CORPORA,"musan/noise/**/*.wav"), label="MUSAN noise")probe(os.path.join(CORPORA,"RIRS_NOISES/real_rirs_isotropic_noises/*.wav"), label="RIRS real")probe(os.path.join(MAD_DIR,"**/*.wav"), label="MAD military")

## 4. Build the RHEAR mixtures`clean speech + real noise → noisy speech`, with the clean speech preserved asthe target.Splits are **speaker-disjoint, noise-source-disjoint and room-disjoint byconstruction**, and a separate **held-out RHEAR real-world test set** is writtenthat is never used for training or tuning.

In [ ]:
SPEECH_ROOTS = {    "train": os.path.join(CORPORA, "LibriSpeech",                          "train-clean-100" if PROFILE=="full" else "dev-clean"),    "val":   os.path.join(CORPORA, "LibriSpeech", "dev-clean"),    "test":  os.path.join(CORPORA, "LibriSpeech", "test-clean"),}N = {"train": 4000, "val": 500, "test": 500} if PROFILE=="full" else \    {"train": 1500, "val": 250, "test": 250}t0 = time.time()n, audit = build.build(    out_dir=OUT, speech_roots=SPEECH_ROOTS,    musan_root=os.path.join(CORPORA, "musan"),    mad_root=MAD_DIR if os.path.isdir(MAD_DIR) else None,    rir_root=os.path.join(CORPORA, "RIRS_NOISES"),    n_per_split=N, dur_s=4.0, seed=1234, snr_range=(-10.0, 20.0), heldout_n=300)print(f"built {n} samples in {(time.time()-t0)/60:.1f} min")

### 4b. Noise-source grouping — inspect before trusting the split

Splits are made at the SOURCE RECORDING level. For MAD, clips are segmented from
source videos, so the grouping depends on the filename pattern. Two failure modes:

* **too few groups** — a whole class collapses into one group and lands entirely in
  one split;
* **one group per clip** — clips cut from the same video land in different splits and
  leak in disguised form.

Neither is detectable from the leakage audit alone, so it is checked explicitly.

In [ ]:
from rhear_data import splits as S
import glob
for corpus, pat in [("musan", os.path.join(CORPORA,"musan/noise/**/*.wav")),
                    ("mad",   os.path.join(MAD_DIR,"**/*.wav"))]:
    paths = glob.glob(pat, recursive=True)
    if paths:
        print(corpus, json.dumps(S.describe_grouping(paths, corpus), indent=1))
    else:
        print(corpus, "-- no files found")

## 5. Leakage auditSpeaker, noise-source and room overlap between splits must all be empty.

In [ ]:
print(json.dumps(audit, indent=2)[:2500])assert audit["clean"], "LEAKAGE DETECTED - do not train on this dataset"print("\nLEAKAGE AUDIT: CLEAN")

## 6. DATASET REPORTEvery number below is computed from the manifest and the audio on disk.

In [ ]:
hdr, rows = manifest.read_manifest(os.path.join(OUT, "manifest.jsonl"))rep = report.compute(hdr, rows, audio_root=OUT)txt = report.render(rep, hdr)print(txt)open(os.path.join(OUT, "DATASET_REPORT.txt"), "w").write(txt)json.dump(rep, open(os.path.join(OUT, "dataset_report.json"), "w"), indent=2)

## 7. Inspect 16 random examplesWaveform and spectrogram, clean vs noisy, with the exact recipe printed.

In [ ]:
import matplotlib.pyplot as pltfrom scipy import signal as sgrng = np.random.default_rng(0)picks = rng.choice(len(rows), size=16, replace=False)fig, ax = plt.subplots(8, 4, figsize=(19, 26))for k, idx in enumerate(picks):    r = rows[idx]    c, _ = sf.read(os.path.join(OUT, r["clean"]))    x, fs = sf.read(os.path.join(OUT, r["noisy"]))    row, col = divmod(k, 2)    a = ax[row][col*2]; b = ax[row][col*2+1]    t = np.arange(len(c))/fs    a.plot(t, c, lw=.5, color="#2b8a3e"); a.plot(t, x, lw=.4, alpha=.55, color="#c92a2a")    cls = ",".join(l["cls"] for l in r["noise_layers"])    a.set_title(f"{r['id']}  SNR {r['snr_db']:+.1f} dB  spk {r['speaker']}\n"                f"{cls} | {r['mixture_stationarity']} | {r['rir_kind']}", fontsize=7)    a.set_xticks([]); a.set_yticks([])    f_, t_, S = sg.spectrogram(x, fs, nperseg=256, noverlap=192)    b.pcolormesh(t_, f_, 10*np.log10(S+1e-10), shading="gouraud", cmap="magma")    b.set_title("noisy spectrogram", fontsize=7); b.set_xticks([]); b.set_yticks([])plt.tight_layout(); plt.savefig(os.path.join(OUT, "examples.png"), dpi=90); plt.show()

In [ ]:
# listen to a fewfrom IPython.display import Audio, displayfor idx in picks[:3]:    r = rows[idx]    print(f"{r['id']}  SNR {r['snr_db']:+.1f} dB  "          f"{[l['cls'] for l in r['noise_layers']]}  traj "          f"{[l['level_traj'] for l in r['noise_layers']]}")    display(Audio(os.path.join(OUT, r["noisy"])))    display(Audio(os.path.join(OUT, r["clean"])))

## 8. ReproducibilityThe manifest header records the seed, library versions, git revision and everysource corpus with its licence. Re-running with the same config reproduces thedataset bit for bit.

In [ ]:
print(json.dumps(hdr, indent=2)[:1800])

## 9. NextDo **not** train yet. Read `E03/model_spec.md` for the proposed L1 baseline —architecture, parameter count, MAC/s, latency, loss and estimated Colab runtime —and approve it before any long run.